# IIR Filter on IQ Signal using GNU Radio

This notebook implements an IIR low-pass filter on a synthetic IQ signal using GNU Radio blocks.

**Workflow:**
1. Generate a synthetic IQ signal (N examples × L samples)
2. Build a GNU Radio flowgraph with `vector_source_c` → `single_pole_iir_filter_cc` → `vector_sink_c`
3. Run the flowgraph
4. Compare signal statistics before and after filtering
5. Verify output shape matches input shape

## 1. Imports

In [ ]:
import numpy as np
from gnuradio import gr, blocks
import matplotlib.pyplot as plt

## 2. Synthetic IQ Signal Generation

Shape convention: `(N, 2, L)` where axis 0 = examples, axis 1 = I/Q, axis 2 = time samples.
We flatten it to a 1-D complex vector for GNU Radio.

In [ ]:
N    = 5       # number of examples
L    = 1000    # samples per example
SEED = 42

rng = np.random.default_rng(SEED)

# Shape (N, 2, L): axis1=0 is I, axis1=1 is Q
iq_tensor = rng.standard_normal((N, 2, L)).astype(np.float32)

# Combine into complex (N, L)
iq_complex = iq_tensor[:, 0, :] + 1j * iq_tensor[:, 1, :]  # shape (N, L)

print(f'IQ tensor shape : {iq_tensor.shape}  (N, 2, L)')
print(f'Complex shape   : {iq_complex.shape}  (N, L)')
print(f'Mean magnitude  : {np.abs(iq_complex).mean():.4f}')
print(f'Std  magnitude  : {np.abs(iq_complex).std():.4f}')

## 3. GNU Radio Flowgraph

```
vector_source_c  →  single_pole_iir_filter_cc  →  vector_sink_c
```

`alpha` controls the filter bandwidth: values closer to 0 give heavier smoothing (narrower bandwidth).

In [ ]:
class IIRFilterFlowgraph(gr.top_block):
    """GNU Radio top block: single-pole IIR filter on complex IQ data."""

    def __init__(self, iq_flat: list, alpha: float = 0.1):
        gr.top_block.__init__(self, 'IIR Filter Flowgraph')

        # Source: complex vector (repeat=False → run once through all samples)
        self.source = blocks.vector_source_c(iq_flat, repeat=False)

        # IIR filter: single-pole low-pass on complex samples
        # GNU Radio applies the same alpha to both I and Q channels
        self.iir = blocks.single_pole_iir_filter_cc(alpha)

        # Sink: collects output samples
        self.sink = blocks.vector_sink_c()

        # Connect blocks
        self.connect(self.source, self.iir, self.sink)


# Prepare flat complex list for GNU Radio
ALPHA    = 0.1
iq_flat  = iq_complex.flatten().tolist()

fg = IIRFilterFlowgraph(iq_flat, alpha=ALPHA)
print(f'Flowgraph built | alpha={ALPHA} | input samples={len(iq_flat)}')

## 4. Run the Flowgraph

In [ ]:
fg.run()

output_flat = np.array(fg.sink.data(), dtype=np.complex64)
print(f'Input  samples : {len(iq_flat)}')
print(f'Output samples : {len(output_flat)}')

## 5. Verify Shape and Statistics

In [ ]:
# Shape check
assert len(output_flat) == len(iq_flat), \
    f'Shape mismatch: input {len(iq_flat)} vs output {len(output_flat)}'
print('Shape check PASSED')

# Reshape back to (N, L)
output_complex = output_flat.reshape(N, L)

# Statistics
print()
print('--- Signal Statistics ---')
print(f'Input  | mean mag: {np.abs(iq_complex).mean():.4f} | std: {np.abs(iq_complex).std():.4f}')
print(f'Output | mean mag: {np.abs(output_complex).mean():.4f} | std: {np.abs(output_complex).std():.4f}')
print()
print('IIR smoothing reduces variance (std decreases) — expected behaviour.')

## 6. Plot: Input vs Filtered Signal (Example 0, I channel)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

t = np.arange(L)

axes[0].plot(t, iq_complex[0].real, color='steelblue', linewidth=0.8, label='I channel (input)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Input IQ Signal — Example 0, I channel')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, output_complex[0].real, color='tomato', linewidth=0.8, label=f'I channel (IIR filtered, alpha={ALPHA})')
axes[1].set_xlabel('Sample index')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('Filtered IQ Signal — Example 0, I channel')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Summary

| Step | Detail |
|------|--------|
| Signal shape | `(N=5, 2, L=1000)` → flattened to `N×L = 5000` complex samples |
| GNU Radio block | `single_pole_iir_filter_cc` with `alpha=0.1` |
| Filter effect | Low-pass smoothing; std of magnitude decreases |
| Shape verified | Output length equals input length |

**Next steps:**
- Tune `alpha` and observe frequency response changes
- Replace synthetic signal with real IQ captures
- Add frequency-domain analysis (FFT before/after)